In [ ]:
import os
import numpy as np
import pandas as pd
import warnings

from sklearn.metrics import f1_score, average_precision_score

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# =========================================================
# 0. 修改这里：你的 npz 路径
# =========================================================

npz_path = "results/trial_scpnet_hill_Ignore_sp_rebuttal/raw_predictions_110cls_epoch_3_mAP_6.03_loss_64.5186_ema.npz"

data = np.load(npz_path, allow_pickle=True)

preds = data["preds"]          # [N, 110], sigmoid probabilities
labels = data["labels"]        # [N, 110], GT labels
video_ids = data["video_ids"]  # [N]

labels = np.round(labels)

print("Loaded:", npz_path)
print("preds:", preds.shape)
print("labels:", labels.shape)
print("video_ids:", video_ids.shape)


# =========================================================
# 1. Helper functions
# =========================================================

def dataset_mask(video_ids, name):
    """
    Follow your calculate_metrics() logic:
    mask = np.array([dataset in v for v in video_ids])
    """
    return np.array([name in str(v) for v in video_ids])


def resolve_nan_like_test(x):
    """
    Same style as your test code: replace -0.0 with NaN.
    """
    x = np.array(x, dtype=float)
    x[x == -0.0] = np.nan
    return x


def safe_average_precision(y_true, y_score):
    """
    AP is undefined if there is no positive sample.
    Return NaN in that case.
    """
    if np.sum(y_true) == 0:
        return np.nan
    return average_precision_score(y_true, y_score) * 100


# =========================================================
# 2. Cholec80: per-phase F1
#    Filter cholec80 first, then use [:, :7]
#    Main metric is per-video macro F1.
#    Here we extend it to per-video per-class F1, then nanmean over videos.
# =========================================================

def cholec80_per_phase_detail(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "cholec80")

    true = labels[mask][:, :7]
    pred = preds[mask][:, :7]
    vids = video_ids[mask]

    print(f"Cholec80 samples: {len(vids)}")

    unique_vids = np.unique(vids)
    per_video_class_f1 = []

    for vid in unique_vids:
        idx = np.where(vids == vid)[0]

        y_true_video = np.argmax(true[idx], axis=1)
        y_pred_video = np.argmax(pred[idx], axis=1)

        cls_f1 = np.full(7, np.nan)

        # Only compute a class F1 for a video if that GT class appears in that video.
        # This follows the spirit of your original video-level macro F1,
        # where labels=np.unique(y_true_video).
        for cls in range(7):
            if np.any(y_true_video == cls):
                cls_f1[cls] = f1_score(
                    y_true_video,
                    y_pred_video,
                    average=None,
                    labels=[cls]
                )[0] * 100

        per_video_class_f1.append(cls_f1.reshape(1, -1))

    per_video_class_f1 = np.concatenate(per_video_class_f1, axis=0)
    mean_per_class_f1 = np.nanmean(per_video_class_f1, axis=0)

    y_true_all = np.argmax(true, axis=1)

    rows = []
    for cls in range(7):
        rows.append({
            "task": "cholec80",
            "class_idx": cls,
            "frequency": int((y_true_all == cls).sum()),
            "metric": "F1",
            "score": float(mean_per_class_f1[cls]),
        })

    return pd.DataFrame(rows)


# =========================================================
# 3. Endoscapes: per-CVS-label AP
#    Filter endoscapes first, then use [:, 7:10]
#    This matches your process_endo() per_class_map logic.
# =========================================================

def endoscapes_per_class_detail(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "endoscapes")

    true = labels[mask][:, 7:10]
    pred = preds[mask][:, 7:10]

    print(f"Endoscapes samples: {true.shape[0]}")

    rows = []
    for cls in range(3):
        freq = int(true[:, cls].sum())
        ap = safe_average_precision(true[:, cls], pred[:, cls])

        rows.append({
            "task": "endoscapes",
            "class_idx": cls,
            "frequency": freq,
            "metric": "AP",
            "score": float(ap) if not np.isnan(ap) else np.nan,
        })

    return pd.DataFrame(rows)


# =========================================================
# 4. CholecT50: IVT only, 100 triplet classes
#    Filter cholect50 first, then use [:, 10:]
#    Same style as process_cholect50():
#       - compute AP per video with average=None
#       - nanmean over videos
#    Only keep IVT triplet APs because you only need rare triplet labels.
# =========================================================

def cholect50_ivt_detail(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "cholect50")

    true = labels[mask][:, 10:]
    pred = preds[mask][:, 10:]
    vids = video_ids[mask]

    print(f"CholecT50 samples: {true.shape[0]}")

    unique_vids = np.unique(vids)
    ap_ivt_list = []

    for vid in unique_vids:
        idx = np.where(vids == vid)[0]

        ivt_labels = true[idx]
        ivt_preds = pred[idx]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ap_ivt = average_precision_score(
                ivt_labels,
                ivt_preds,
                average=None
            ) * 100

        ap_ivt = resolve_nan_like_test(ap_ivt)
        ap_ivt_list.append(ap_ivt.reshape(1, -1))

    ap_ivt_arr = np.concatenate(ap_ivt_list, axis=0)
    mean_ap_ivt = np.nanmean(ap_ivt_arr, axis=0)

    freq_arr = true.sum(axis=0)

    rows = []
    for cls in range(len(mean_ap_ivt)):
        freq = int(freq_arr[cls])

        # Filter out triplets that do not appear in the CholecT50 test split.
        if freq == 0:
            continue

        rows.append({
            "task": "cholect50",
            "component": "ivt",
            "class_idx": cls,
            "frequency": freq,
            "metric": "AP",
            "score": float(mean_ap_ivt[cls]),
        })

    return pd.DataFrame(rows)


# =========================================================
# 5. Overall sanity checks
# =========================================================

def cholec80_overall_like_test(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "cholec80")

    true = labels[mask][:, :7]
    pred = preds[mask][:, :7]
    vids = video_ids[mask]

    unique_vids = np.unique(vids)
    video_f1s = []

    for vid in unique_vids:
        idx = np.where(vids == vid)[0]

        y_true_video = np.argmax(true[idx], axis=1)
        y_pred_video = np.argmax(pred[idx], axis=1)

        f1 = f1_score(
            y_true_video,
            y_pred_video,
            average="macro",
            labels=np.unique(y_true_video)
        ) * 100

        video_f1s.append(f1)

    return float(np.mean(video_f1s))


def endoscapes_overall_like_test(df_endo):
    return float(df_endo["score"].mean())


def cholect50_ivt_overall_like_test(df_t50_ivt):
    return float(df_t50_ivt["score"].mean())


# =========================================================
# 6. Run analysis
# =========================================================

df_phase = cholec80_per_phase_detail(labels, preds, video_ids)
df_endo = endoscapes_per_class_detail(labels, preds, video_ids)
df_t50_ivt = cholect50_ivt_detail(labels, preds, video_ids)

print("\n========== Cholec80 per-phase F1 ==========")
display(df_phase)

print("\n========== Endoscapes per-CVS AP ==========")
display(df_endo)

print("\n========== CholecT50 IVT per-triplet AP ==========")
display(df_t50_ivt)


# =========================================================
# 7. Sanity checks
# =========================================================

print("\n========== Sanity checks ==========")

cholec80_f1 = cholec80_overall_like_test(labels, preds, video_ids)
endo_map = endoscapes_overall_like_test(df_endo)
t50_ap_ivt = cholect50_ivt_overall_like_test(df_t50_ivt)

print("Cholec80 overall F1 like test:", cholec80_f1)
print("Endoscapes mAP from per-class AP:", endo_map)
print("CholecT50 AP-IVT from per-triplet AP:", t50_ap_ivt)


# =========================================================
# 8. Rare label views
# =========================================================

print("\n========== Rare Cholec80 phases ==========")
display(df_phase.sort_values("frequency"))

print("\n========== Rare Endoscapes CVS labels ==========")
display(df_endo.sort_values("frequency"))

print("\n========== Rare CholecT50 IVT triplets ==========")
display(df_t50_ivt.sort_values("frequency").head(20))


# =========================================================
# 9. Hardest label views
# =========================================================

print("\n========== Hardest Cholec80 phases ==========")
display(df_phase.sort_values("score").head(7))

print("\n========== Hardest Endoscapes CVS labels ==========")
display(df_endo.sort_values("score").head(3))

print("\n========== Hardest CholecT50 IVT triplets ==========")
display(df_t50_ivt.sort_values("score").head(20))


# =========================================================
# 10. Save CSV
# =========================================================

out_dir = "rebuttal_metric_analysis"
os.makedirs(out_dir, exist_ok=True)

df_phase.to_csv(
    os.path.join(out_dir, "cholec80_per_phase_f1.csv"),
    index=False
)

df_endo.to_csv(
    os.path.join(out_dir, "endoscapes_per_cvs_ap.csv"),
    index=False
)

df_t50_ivt.to_csv(
    os.path.join(out_dir, "cholect50_ivt_per_triplet_ap.csv"),
    index=False
)

print("\nSaved CSVs to:", out_dir)


# =========================================================
# 11. Print selected rare-label metrics for rebuttal
# =========================================================

print("\n\n==============================")
print("Selected rare-label metrics")
print("==============================")

# -----------------------------
# Cholec80: lowest-frequency 2 phases
# -----------------------------
phase_rare2 = (
    df_phase
    .sort_values("frequency", ascending=True)
    .head(2)
    .copy()
)

print("\n[Cholec80] Lowest-frequency 2 phases")
display(phase_rare2)

print(
    "[Cholec80] Rare-2 mean F1:",
    phase_rare2["score"].mean()
)


# -----------------------------
# Endoscapes: all 3 CVS labels sorted by frequency
# -----------------------------
endo_by_freq = (
    df_endo
    .sort_values("frequency", ascending=True)
    .copy()
)

print("\n[Endoscapes] CVS labels sorted by frequency")
display(endo_by_freq)

print(
    "[Endoscapes] Mean AP over 3 CVS labels:",
    endo_by_freq["score"].mean()
)


# -----------------------------
# CholecT50: lowest-frequency 10 IVT triplets
# -----------------------------
t50_rare10 = (
    df_t50_ivt
    .sort_values("frequency", ascending=True)
    .head(10)
    .copy()
)

print("\n[CholecT50] Lowest-frequency 10 IVT triplets")
display(t50_rare10)

print(
    "[CholecT50] Rare-10 mean AP:",
    t50_rare10["score"].mean()
)


# -----------------------------
# Also print compact summary table
# -----------------------------
summary_rows = [
    {
        "dataset": "Cholec80",
        "selected_labels": "lowest-frequency 2 phases",
        "metric": "F1",
        "num_labels": len(phase_rare2),
        "mean_score": phase_rare2["score"].mean(),
        "mean_frequency": phase_rare2["frequency"].mean(),
    },
    {
        "dataset": "Endoscapes",
        "selected_labels": "all 3 CVS labels",
        "metric": "AP",
        "num_labels": len(endo_by_freq),
        "mean_score": endo_by_freq["score"].mean(),
        "mean_frequency": endo_by_freq["frequency"].mean(),
    },
    {
        "dataset": "CholecT50",
        "selected_labels": "lowest-frequency 10 IVT triplets",
        "metric": "AP",
        "num_labels": len(t50_rare10),
        "mean_score": t50_rare10["score"].mean(),
        "mean_frequency": t50_rare10["frequency"].mean(),
    },
]

summary_rare = pd.DataFrame(summary_rows)

print("\n[Summary] Rare-label selected groups")
display(summary_rare)


# -----------------------------
# Save selected rare-label tables
# -----------------------------
phase_rare2.to_csv(
    os.path.join(out_dir, "selected_cholec80_lowest_freq_2_phases.csv"),
    index=False
)

endo_by_freq.to_csv(
    os.path.join(out_dir, "selected_endoscapes_all_3_by_frequency.csv"),
    index=False
)

t50_rare10.to_csv(
    os.path.join(out_dir, "selected_cholect50_lowest_freq_10_ivt.csv"),
    index=False
)

summary_rare.to_csv(
    os.path.join(out_dir, "selected_rare_label_summary.csv"),
    index=False
)

print("\nSaved selected rare-label CSVs to:", out_dir)

# =========================================================
# 12. Frequency-group summary: Low / Mid / High
# =========================================================

def make_frequency_group_summary(df, dataset_name, metric_name, group_col=None):
    """
    按 frequency 从低到高排序，然后分成 Low / Mid / High 三组。
    输出每组的平均 score、平均 frequency、frequency range。
    """

    df = df.copy()
    df = df.sort_values("frequency", ascending=True).reset_index(drop=True)

    groups = np.array_split(df, 3)
    group_names = ["Low", "Mid", "High"]

    rows = []

    for group_name, g in zip(group_names, groups):
        if len(g) == 0:
            continue

        row = {
            "dataset": dataset_name,
            "freq_group": group_name,
            "num_labels": len(g),
            "freq_min": int(g["frequency"].min()),
            "freq_max": int(g["frequency"].max()),
            "mean_frequency": float(g["frequency"].mean()),
            "metric": metric_name,
            "mean_score": float(g["score"].mean()),
        }

        if group_col is not None and group_col in g.columns:
            row["component"] = g[group_col].iloc[0]

        rows.append(row)

    return pd.DataFrame(rows)


# -----------------------------
# Cholec80: 7 phases -> Low/Mid/High
# -----------------------------
summary_phase_groups = make_frequency_group_summary(
    df_phase,
    dataset_name="Cholec80",
    metric_name="F1"
)

# -----------------------------
# Endoscapes: 3 CVS labels -> Low/Mid/High
# 每组 1 个 label
# -----------------------------
summary_endo_groups = make_frequency_group_summary(
    df_endo,
    dataset_name="Endoscapes",
    metric_name="AP"
)

# -----------------------------
# CholecT50: IVT triplets only
# 只看有效出现过的 triplet labels
# -----------------------------
summary_t50_groups = make_frequency_group_summary(
    df_t50_ivt,
    dataset_name="CholecT50-IVT",
    metric_name="AP",
    group_col="component"
)


# -----------------------------
# Combine all tasks
# -----------------------------
summary_freq_groups = pd.concat(
    [
        summary_phase_groups,
        summary_endo_groups,
        summary_t50_groups,
    ],
    ignore_index=True
)

print("\n========== Frequency-group summary: Low / Mid / High ==========")
display(summary_freq_groups)


# -----------------------------
# Save
# -----------------------------
summary_freq_groups.to_csv(
    os.path.join(out_dir, "frequency_group_summary_low_mid_high.csv"),
    index=False
)

print("\nSaved frequency-group summary to:", os.path.join(out_dir, "frequency_group_summary_low_mid_high.csv"))


# =========================================================
# 13. Single-model video-level bootstrap confidence intervals
#     Use existing test predictions only; no retraining/inference.
# =========================================================

BOOTSTRAP_N = 1000
BOOTSTRAP_SEED = 20260513


def get_video_index_dict(video_ids_, dataset_name):
    mask = dataset_mask(video_ids_, dataset_name)
    global_idx = np.where(mask)[0]
    vids = np.array([str(v) for v in video_ids_[mask]])

    idx_by_vid = {}
    for vid in np.unique(vids):
        idx_by_vid[vid] = global_idx[vids == vid]

    return idx_by_vid


def summarize_single_model_bootstrap(metric_name, dataset_name, point_score, boot_scores, n_units):
    boot_scores = np.asarray(boot_scores, dtype=float)
    ci_low, ci_mid, ci_high = np.percentile(boot_scores, [2.5, 50, 97.5])

    return {
        "dataset": dataset_name,
        "metric": metric_name,
        "point_score": float(point_score),
        "n_units": int(n_units),
        "bootstrap_resamples": int(len(boot_scores)),
        "ci_2.5": float(ci_low),
        "ci_50": float(ci_mid),
        "ci_97.5": float(ci_high),
    }


# -----------------------------
# 13.1 Cholec80 Phase F1
# Metric matches cholec80_overall_like_test():
# per-video macro F1, then mean over videos.
# Bootstrap unit = video.
# -----------------------------

def cholec80_per_video_f1_values(labels_, preds_, video_ids_):
    idx_by_vid = get_video_index_dict(video_ids_, "cholec80")
    videos = np.array(sorted(idx_by_vid.keys()))

    values = []

    for vid in videos:
        idx = idx_by_vid[vid]

        true = labels_[idx][:, :7]
        pred = preds_[idx][:, :7]

        y_true_video = np.argmax(true, axis=1)
        y_pred_video = np.argmax(pred, axis=1)

        f1 = f1_score(
            y_true_video,
            y_pred_video,
            average="macro",
            labels=np.unique(y_true_video),
        ) * 100

        values.append(float(f1))

    return videos, np.asarray(values, dtype=float)


def bootstrap_cholec80_phase_f1_single(labels_, preds_, video_ids_, rng):
    videos, per_video_f1 = cholec80_per_video_f1_values(labels_, preds_, video_ids_)

    point_score = float(np.mean(per_video_f1))

    boot_scores = []
    for _ in range(BOOTSTRAP_N):
        sample_idx = rng.integers(0, len(videos), size=len(videos))
        boot_scores.append(float(np.mean(per_video_f1[sample_idx])))

    return summarize_single_model_bootstrap(
        metric_name="Phase F1",
        dataset_name="Cholec80",
        point_score=point_score,
        boot_scores=boot_scores,
        n_units=len(videos),
    )


# -----------------------------
# 13.2 Endoscapes CVS mAP
# Metric matches endoscapes_per_class_detail():
# AP for C1/C2/C3 over all Endoscapes frames, then mean.
# Bootstrap unit = video; sampled videos' frames are concatenated.
# -----------------------------

def endoscapes_map_for_indices(labels_, preds_, idx):
    true = labels_[idx][:, 7:10]
    pred = preds_[idx][:, 7:10]

    aps = []
    for cls in range(3):
        aps.append(safe_average_precision(true[:, cls], pred[:, cls]))

    return float(np.nanmean(aps))


def bootstrap_endoscapes_cvs_map_single(labels_, preds_, video_ids_, rng):
    idx_by_vid = get_video_index_dict(video_ids_, "endoscapes")
    videos = np.array(sorted(idx_by_vid.keys()))

    all_idx = np.concatenate([idx_by_vid[v] for v in videos])
    point_score = endoscapes_map_for_indices(labels_, preds_, all_idx)

    boot_scores = []
    for _ in range(BOOTSTRAP_N):
        sample_videos = rng.choice(videos, size=len(videos), replace=True)
        sample_idx = np.concatenate([idx_by_vid[v] for v in sample_videos])
        boot_scores.append(endoscapes_map_for_indices(labels_, preds_, sample_idx))

    return summarize_single_model_bootstrap(
        metric_name="CVS mAP",
        dataset_name="Endoscapes",
        point_score=point_score,
        boot_scores=boot_scores,
        n_units=len(videos),
    )


# -----------------------------
# 13.3 CholecT50 AP-IVT
# Metric matches your cholect50_ivt_detail():
# per-video AP vector -> nanmean over videos -> mean over valid IVT labels.
# Bootstrap unit = video.
# -----------------------------

def cholect50_per_video_ivt_ap_matrix(labels_, preds_, video_ids_):
    idx_by_vid = get_video_index_dict(video_ids_, "cholect50")
    videos = np.array(sorted(idx_by_vid.keys()))

    all_idx = np.concatenate([idx_by_vid[v] for v in videos])
    global_freq = labels_[all_idx][:, 10:].sum(axis=0)
    valid_classes = global_freq > 0

    ap_rows = []

    for vid in videos:
        idx = idx_by_vid[vid]

        true = labels_[idx][:, 10:]
        pred = preds_[idx][:, 10:]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ap_vec = average_precision_score(
                true,
                pred,
                average=None,
            ) * 100

        ap_vec = resolve_nan_like_test(ap_vec)
        ap_rows.append(ap_vec.reshape(1, -1))

    ap_matrix = np.concatenate(ap_rows, axis=0)
    return videos, ap_matrix, valid_classes


def cholect50_ivt_from_ap_matrix(ap_matrix, valid_classes):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        class_means = np.nanmean(ap_matrix[:, valid_classes], axis=0)
        return float(np.nanmean(class_means))


def bootstrap_cholect50_ivt_single(labels_, preds_, video_ids_, rng):
    videos, ap_matrix, valid_classes = cholect50_per_video_ivt_ap_matrix(
        labels_,
        preds_,
        video_ids_,
    )

    point_score = cholect50_ivt_from_ap_matrix(ap_matrix, valid_classes)

    boot_scores = []
    for _ in range(BOOTSTRAP_N):
        sample_idx = rng.integers(0, len(videos), size=len(videos))
        boot_scores.append(
            cholect50_ivt_from_ap_matrix(ap_matrix[sample_idx], valid_classes)
        )

    return


In [ ]:
import os
import numpy as np
import pandas as pd
import warnings

from sklearn.metrics import f1_score, average_precision_score

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# =========================================================
# 0. 修改这里：你的 npz 路径
# =========================================================

npz_path = "/data/MML1209/results/trial_scpnet_hill_Ignore_sp-comp-rebuttal/raw_predictions_110cls_epoch_3_mAP_6.11_loss_64.3053_ema.npz"

data = np.load(npz_path, allow_pickle=True)

preds = data["preds"]          # [N, 110], sigmoid probabilities
labels = data["labels"]        # [N, 110], GT labels
video_ids = data["video_ids"]  # [N]

labels = np.round(labels)

print("Loaded:", npz_path)
print("preds:", preds.shape)
print("labels:", labels.shape)
print("video_ids:", video_ids.shape)


# =========================================================
# 1. Helper functions
# =========================================================

def dataset_mask(video_ids, name):
    """
    Follow your calculate_metrics() logic:
    mask = np.array([dataset in v for v in video_ids])
    """
    return np.array([name in str(v) for v in video_ids])


def resolve_nan_like_test(x):
    """
    Same style as your test code: replace -0.0 with NaN.
    """
    x = np.array(x, dtype=float)
    x[x == -0.0] = np.nan
    return x


def safe_average_precision(y_true, y_score):
    """
    AP is undefined if there is no positive sample.
    Return NaN in that case.
    """
    if np.sum(y_true) == 0:
        return np.nan
    return average_precision_score(y_true, y_score) * 100


# =========================================================
# 2. Cholec80: per-phase F1
#    Filter cholec80 first, then use [:, :7]
#    Main metric is per-video macro F1.
#    Here we extend it to per-video per-class F1, then nanmean over videos.
# =========================================================

def cholec80_per_phase_detail(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "cholec80")

    true = labels[mask][:, :7]
    pred = preds[mask][:, :7]
    vids = video_ids[mask]

    print(f"Cholec80 samples: {len(vids)}")

    unique_vids = np.unique(vids)
    per_video_class_f1 = []

    for vid in unique_vids:
        idx = np.where(vids == vid)[0]

        y_true_video = np.argmax(true[idx], axis=1)
        y_pred_video = np.argmax(pred[idx], axis=1)

        cls_f1 = np.full(7, np.nan)

        # Only compute a class F1 for a video if that GT class appears in that video.
        # This follows the spirit of your original video-level macro F1,
        # where labels=np.unique(y_true_video).
        for cls in range(7):
            if np.any(y_true_video == cls):
                cls_f1[cls] = f1_score(
                    y_true_video,
                    y_pred_video,
                    average=None,
                    labels=[cls]
                )[0] * 100

        per_video_class_f1.append(cls_f1.reshape(1, -1))

    per_video_class_f1 = np.concatenate(per_video_class_f1, axis=0)
    mean_per_class_f1 = np.nanmean(per_video_class_f1, axis=0)

    y_true_all = np.argmax(true, axis=1)

    rows = []
    for cls in range(7):
        rows.append({
            "task": "cholec80",
            "class_idx": cls,
            "frequency": int((y_true_all == cls).sum()),
            "metric": "F1",
            "score": float(mean_per_class_f1[cls]),
        })

    return pd.DataFrame(rows)


# =========================================================
# 3. Endoscapes: per-CVS-label AP
#    Filter endoscapes first, then use [:, 7:10]
#    This matches your process_endo() per_class_map logic.
# =========================================================

def endoscapes_per_class_detail(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "endoscapes")

    true = labels[mask][:, 7:10]
    pred = preds[mask][:, 7:10]

    print(f"Endoscapes samples: {true.shape[0]}")

    rows = []
    for cls in range(3):
        freq = int(true[:, cls].sum())
        ap = safe_average_precision(true[:, cls], pred[:, cls])

        rows.append({
            "task": "endoscapes",
            "class_idx": cls,
            "frequency": freq,
            "metric": "AP",
            "score": float(ap) if not np.isnan(ap) else np.nan,
        })

    return pd.DataFrame(rows)


# =========================================================
# 4. CholecT50: IVT only, 100 triplet classes
#    Filter cholect50 first, then use [:, 10:]
#    Same style as process_cholect50():
#       - compute AP per video with average=None
#       - nanmean over videos
#    Only keep IVT triplet APs because you only need rare triplet labels.
# =========================================================

def cholect50_ivt_detail(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "cholect50")

    true = labels[mask][:, 10:]
    pred = preds[mask][:, 10:]
    vids = video_ids[mask]

    print(f"CholecT50 samples: {true.shape[0]}")

    unique_vids = np.unique(vids)
    ap_ivt_list = []

    for vid in unique_vids:
        idx = np.where(vids == vid)[0]

        ivt_labels = true[idx]
        ivt_preds = pred[idx]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ap_ivt = average_precision_score(
                ivt_labels,
                ivt_preds,
                average=None
            ) * 100

        ap_ivt = resolve_nan_like_test(ap_ivt)
        ap_ivt_list.append(ap_ivt.reshape(1, -1))

    ap_ivt_arr = np.concatenate(ap_ivt_list, axis=0)
    mean_ap_ivt = np.nanmean(ap_ivt_arr, axis=0)

    freq_arr = true.sum(axis=0)

    rows = []
    for cls in range(len(mean_ap_ivt)):
        freq = int(freq_arr[cls])

        # Filter out triplets that do not appear in the CholecT50 test split.
        if freq == 0:
            continue

        rows.append({
            "task": "cholect50",
            "component": "ivt",
            "class_idx": cls,
            "frequency": freq,
            "metric": "AP",
            "score": float(mean_ap_ivt[cls]),
        })

    return pd.DataFrame(rows)


# =========================================================
# 5. Overall sanity checks
# =========================================================

def cholec80_overall_like_test(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "cholec80")

    true = labels[mask][:, :7]
    pred = preds[mask][:, :7]
    vids = video_ids[mask]

    unique_vids = np.unique(vids)
    video_f1s = []

    for vid in unique_vids:
        idx = np.where(vids == vid)[0]

        y_true_video = np.argmax(true[idx], axis=1)
        y_pred_video = np.argmax(pred[idx], axis=1)

        f1 = f1_score(
            y_true_video,
            y_pred_video,
            average="macro",
            labels=np.unique(y_true_video)
        ) * 100

        video_f1s.append(f1)

    return float(np.mean(video_f1s))


def endoscapes_overall_like_test(df_endo):
    return float(df_endo["score"].mean())


def cholect50_ivt_overall_like_test(df_t50_ivt):
    return float(df_t50_ivt["score"].mean())


# =========================================================
# 6. Run analysis
# =========================================================

df_phase = cholec80_per_phase_detail(labels, preds, video_ids)
df_endo = endoscapes_per_class_detail(labels, preds, video_ids)
df_t50_ivt = cholect50_ivt_detail(labels, preds, video_ids)

print("\n========== Cholec80 per-phase F1 ==========")
display(df_phase)

print("\n========== Endoscapes per-CVS AP ==========")
display(df_endo)

print("\n========== CholecT50 IVT per-triplet AP ==========")
display(df_t50_ivt)


# =========================================================
# 7. Sanity checks
# =========================================================

print("\n========== Sanity checks ==========")

cholec80_f1 = cholec80_overall_like_test(labels, preds, video_ids)
endo_map = endoscapes_overall_like_test(df_endo)
t50_ap_ivt = cholect50_ivt_overall_like_test(df_t50_ivt)

print("Cholec80 overall F1 like test:", cholec80_f1)
print("Endoscapes mAP from per-class AP:", endo_map)
print("CholecT50 AP-IVT from per-triplet AP:", t50_ap_ivt)


# =========================================================
# 8. Rare label views
# =========================================================

print("\n========== Rare Cholec80 phases ==========")
display(df_phase.sort_values("frequency"))

print("\n========== Rare Endoscapes CVS labels ==========")
display(df_endo.sort_values("frequency"))

print("\n========== Rare CholecT50 IVT triplets ==========")
display(df_t50_ivt.sort_values("frequency").head(20))


# =========================================================
# 9. Hardest label views
# =========================================================

print("\n========== Hardest Cholec80 phases ==========")
display(df_phase.sort_values("score").head(7))

print("\n========== Hardest Endoscapes CVS labels ==========")
display(df_endo.sort_values("score").head(3))

print("\n========== Hardest CholecT50 IVT triplets ==========")
display(df_t50_ivt.sort_values("score").head(20))


# =========================================================
# 10. Save CSV
# =========================================================

out_dir = "rebuttal_metric_analysis-comp"
os.makedirs(out_dir, exist_ok=True)

df_phase.to_csv(
    os.path.join(out_dir, "cholec80_per_phase_f1.csv"),
    index=False
)

df_endo.to_csv(
    os.path.join(out_dir, "endoscapes_per_cvs_ap.csv"),
    index=False
)

df_t50_ivt.to_csv(
    os.path.join(out_dir, "cholect50_ivt_per_triplet_ap.csv"),
    index=False
)

print("\nSaved CSVs to:", out_dir)

# =========================================================
# 11. Print selected rare-label metrics for rebuttal
# =========================================================

print("\n\n==============================")
print("Selected rare-label metrics")
print("==============================")

# -----------------------------
# Cholec80: lowest-frequency 2 phases
# -----------------------------
phase_rare2 = (
    df_phase
    .sort_values("frequency", ascending=True)
    .head(2)
    .copy()
)

print("\n[Cholec80] Lowest-frequency 2 phases")
display(phase_rare2)

print(
    "[Cholec80] Rare-2 mean F1:",
    phase_rare2["score"].mean()
)


# -----------------------------
# Endoscapes: all 3 CVS labels sorted by frequency
# -----------------------------
endo_by_freq = (
    df_endo
    .sort_values("frequency", ascending=True)
    .copy()
)

print("\n[Endoscapes] CVS labels sorted by frequency")
display(endo_by_freq)

print(
    "[Endoscapes] Mean AP over 3 CVS labels:",
    endo_by_freq["score"].mean()
)


# -----------------------------
# CholecT50: lowest-frequency 10 IVT triplets
# -----------------------------
t50_rare10 = (
    df_t50_ivt
    .sort_values("frequency", ascending=True)
    .head(10)
    .copy()
)

print("\n[CholecT50] Lowest-frequency 10 IVT triplets")
display(t50_rare10)

print(
    "[CholecT50] Rare-10 mean AP:",
    t50_rare10["score"].mean()
)


# -----------------------------
# Also print compact summary table
# -----------------------------
summary_rows = [
    {
        "dataset": "Cholec80",
        "selected_labels": "lowest-frequency 2 phases",
        "metric": "F1",
        "num_labels": len(phase_rare2),
        "mean_score": phase_rare2["score"].mean(),
        "mean_frequency": phase_rare2["frequency"].mean(),
    },
    {
        "dataset": "Endoscapes",
        "selected_labels": "all 3 CVS labels",
        "metric": "AP",
        "num_labels": len(endo_by_freq),
        "mean_score": endo_by_freq["score"].mean(),
        "mean_frequency": endo_by_freq["frequency"].mean(),
    },
    {
        "dataset": "CholecT50",
        "selected_labels": "lowest-frequency 10 IVT triplets",
        "metric": "AP",
        "num_labels": len(t50_rare10),
        "mean_score": t50_rare10["score"].mean(),
        "mean_frequency": t50_rare10["frequency"].mean(),
    },
]

summary_rare = pd.DataFrame(summary_rows)

print("\n[Summary] Rare-label selected groups")
display(summary_rare)


# -----------------------------
# Save selected rare-label tables
# -----------------------------
phase_rare2.to_csv(
    os.path.join(out_dir, "selected_cholec80_lowest_freq_2_phases.csv"),
    index=False
)

endo_by_freq.to_csv(
    os.path.join(out_dir, "selected_endoscapes_all_3_by_frequency.csv"),
    index=False
)

t50_rare10.to_csv(
    os.path.join(out_dir, "selected_cholect50_lowest_freq_10_ivt.csv"),
    index=False
)

summary_rare.to_csv(
    os.path.join(out_dir, "selected_rare_label_summary.csv"),
    index=False
)

print("\nSaved selected rare-label CSVs to:", out_dir)

# =========================================================
# 12. Frequency-group summary: Low / Mid / High
# =========================================================

def make_frequency_group_summary(df, dataset_name, metric_name, group_col=None):
    """
    按 frequency 从低到高排序，然后分成 Low / Mid / High 三组。
    输出每组的平均 score、平均 frequency、frequency range。
    """

    df = df.copy()
    df = df.sort_values("frequency", ascending=True).reset_index(drop=True)

    groups = np.array_split(df, 3)
    group_names = ["Low", "Mid", "High"]

    rows = []

    for group_name, g in zip(group_names, groups):
        if len(g) == 0:
            continue

        row = {
            "dataset": dataset_name,
            "freq_group": group_name,
            "num_labels": len(g),
            "freq_min": int(g["frequency"].min()),
            "freq_max": int(g["frequency"].max()),
            "mean_frequency": float(g["frequency"].mean()),
            "metric": metric_name,
            "mean_score": float(g["score"].mean()),
        }

        if group_col is not None and group_col in g.columns:
            row["component"] = g[group_col].iloc[0]

        rows.append(row)

    return pd.DataFrame(rows)


# -----------------------------
# Cholec80: 7 phases -> Low/Mid/High
# -----------------------------
summary_phase_groups = make_frequency_group_summary(
    df_phase,
    dataset_name="Cholec80",
    metric_name="F1"
)

# -----------------------------
# Endoscapes: 3 CVS labels -> Low/Mid/High
# 每组 1 个 label
# -----------------------------
summary_endo_groups = make_frequency_group_summary(
    df_endo,
    dataset_name="Endoscapes",
    metric_name="AP"
)

# -----------------------------
# CholecT50: IVT triplets only
# 只看有效出现过的 triplet labels
# -----------------------------
summary_t50_groups = make_frequency_group_summary(
    df_t50_ivt,
    dataset_name="CholecT50-IVT",
    metric_name="AP",
    group_col="component"
)


# -----------------------------
# Combine all tasks
# -----------------------------
summary_freq_groups = pd.concat(
    [
        summary_phase_groups,
        summary_endo_groups,
        summary_t50_groups,
    ],
    ignore_index=True
)

print("\n========== Frequency-group summary: Low / Mid / High ==========")
display(summary_freq_groups)


# -----------------------------
# Save
# -----------------------------
summary_freq_groups.to_csv(
    os.path.join(out_dir, "frequency_group_summary_low_mid_high.csv"),
    index=False
)

print("\nSaved frequency-group summary to:", os.path.join(out_dir, "frequency_group_summary_low_mid_high.csv"))

In [ ]:
import os
import numpy as np
import pandas as pd
import warnings

from sklearn.metrics import f1_score, average_precision_score

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# =========================================================
# 0. 修改这里：你的 npz 路径
# =========================================================

npz_path = "/data/MML1209/results/trial_scpnet_hill_Ignore_sp-rebuttal-comp-ignore/raw_predictions_110cls_epoch_3_mAP_5.46_loss_90.7614_ema.npz"

data = np.load(npz_path, allow_pickle=True)

preds = data["preds"]          # [N, 110], sigmoid probabilities
labels = data["labels"]        # [N, 110], GT labels
video_ids = data["video_ids"]  # [N]

labels = np.round(labels)

print("Loaded:", npz_path)
print("preds:", preds.shape)
print("labels:", labels.shape)
print("video_ids:", video_ids.shape)


# =========================================================
# 1. Helper functions
# =========================================================

def dataset_mask(video_ids, name):
    """
    Follow your calculate_metrics() logic:
    mask = np.array([dataset in v for v in video_ids])
    """
    return np.array([name in str(v) for v in video_ids])


def resolve_nan_like_test(x):
    """
    Same style as your test code: replace -0.0 with NaN.
    """
    x = np.array(x, dtype=float)
    x[x == -0.0] = np.nan
    return x


def safe_average_precision(y_true, y_score):
    """
    AP is undefined if there is no positive sample.
    Return NaN in that case.
    """
    if np.sum(y_true) == 0:
        return np.nan
    return average_precision_score(y_true, y_score) * 100


# =========================================================
# 2. Cholec80: per-phase F1
#    Filter cholec80 first, then use [:, :7]
#    Main metric is per-video macro F1.
#    Here we extend it to per-video per-class F1, then nanmean over videos.
# =========================================================

def cholec80_per_phase_detail(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "cholec80")

    true = labels[mask][:, :7]
    pred = preds[mask][:, :7]
    vids = video_ids[mask]

    print(f"Cholec80 samples: {len(vids)}")

    unique_vids = np.unique(vids)
    per_video_class_f1 = []

    for vid in unique_vids:
        idx = np.where(vids == vid)[0]

        y_true_video = np.argmax(true[idx], axis=1)
        y_pred_video = np.argmax(pred[idx], axis=1)

        cls_f1 = np.full(7, np.nan)

        # Only compute a class F1 for a video if that GT class appears in that video.
        # This follows the spirit of your original video-level macro F1,
        # where labels=np.unique(y_true_video).
        for cls in range(7):
            if np.any(y_true_video == cls):
                cls_f1[cls] = f1_score(
                    y_true_video,
                    y_pred_video,
                    average=None,
                    labels=[cls]
                )[0] * 100

        per_video_class_f1.append(cls_f1.reshape(1, -1))

    per_video_class_f1 = np.concatenate(per_video_class_f1, axis=0)
    mean_per_class_f1 = np.nanmean(per_video_class_f1, axis=0)

    y_true_all = np.argmax(true, axis=1)

    rows = []
    for cls in range(7):
        rows.append({
            "task": "cholec80",
            "class_idx": cls,
            "frequency": int((y_true_all == cls).sum()),
            "metric": "F1",
            "score": float(mean_per_class_f1[cls]),
        })

    return pd.DataFrame(rows)


# =========================================================
# 3. Endoscapes: per-CVS-label AP
#    Filter endoscapes first, then use [:, 7:10]
#    This matches your process_endo() per_class_map logic.
# =========================================================

def endoscapes_per_class_detail(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "endoscapes")

    true = labels[mask][:, 7:10]
    pred = preds[mask][:, 7:10]

    print(f"Endoscapes samples: {true.shape[0]}")

    rows = []
    for cls in range(3):
        freq = int(true[:, cls].sum())
        ap = safe_average_precision(true[:, cls], pred[:, cls])

        rows.append({
            "task": "endoscapes",
            "class_idx": cls,
            "frequency": freq,
            "metric": "AP",
            "score": float(ap) if not np.isnan(ap) else np.nan,
        })

    return pd.DataFrame(rows)


# =========================================================
# 4. CholecT50: IVT only, 100 triplet classes
#    Filter cholect50 first, then use [:, 10:]
#    Same style as process_cholect50():
#       - compute AP per video with average=None
#       - nanmean over videos
#    Only keep IVT triplet APs because you only need rare triplet labels.
# =========================================================

def cholect50_ivt_detail(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "cholect50")

    true = labels[mask][:, 10:]
    pred = preds[mask][:, 10:]
    vids = video_ids[mask]

    print(f"CholecT50 samples: {true.shape[0]}")

    unique_vids = np.unique(vids)
    ap_ivt_list = []

    for vid in unique_vids:
        idx = np.where(vids == vid)[0]

        ivt_labels = true[idx]
        ivt_preds = pred[idx]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ap_ivt = average_precision_score(
                ivt_labels,
                ivt_preds,
                average=None
            ) * 100

        ap_ivt = resolve_nan_like_test(ap_ivt)
        ap_ivt_list.append(ap_ivt.reshape(1, -1))

    ap_ivt_arr = np.concatenate(ap_ivt_list, axis=0)
    mean_ap_ivt = np.nanmean(ap_ivt_arr, axis=0)

    freq_arr = true.sum(axis=0)

    rows = []
    for cls in range(len(mean_ap_ivt)):
        freq = int(freq_arr[cls])

        # Filter out triplets that do not appear in the CholecT50 test split.
        if freq == 0:
            continue

        rows.append({
            "task": "cholect50",
            "component": "ivt",
            "class_idx": cls,
            "frequency": freq,
            "metric": "AP",
            "score": float(mean_ap_ivt[cls]),
        })

    return pd.DataFrame(rows)


# =========================================================
# 5. Overall sanity checks
# =========================================================

def cholec80_overall_like_test(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "cholec80")

    true = labels[mask][:, :7]
    pred = preds[mask][:, :7]
    vids = video_ids[mask]

    unique_vids = np.unique(vids)
    video_f1s = []

    for vid in unique_vids:
        idx = np.where(vids == vid)[0]

        y_true_video = np.argmax(true[idx], axis=1)
        y_pred_video = np.argmax(pred[idx], axis=1)

        f1 = f1_score(
            y_true_video,
            y_pred_video,
            average="macro",
            labels=np.unique(y_true_video)
        ) * 100

        video_f1s.append(f1)

    return float(np.mean(video_f1s))


def endoscapes_overall_like_test(df_endo):
    return float(df_endo["score"].mean())


def cholect50_ivt_overall_like_test(df_t50_ivt):
    return float(df_t50_ivt["score"].mean())


# =========================================================
# 6. Run analysis
# =========================================================

df_phase = cholec80_per_phase_detail(labels, preds, video_ids)
df_endo = endoscapes_per_class_detail(labels, preds, video_ids)
df_t50_ivt = cholect50_ivt_detail(labels, preds, video_ids)

print("\n========== Cholec80 per-phase F1 ==========")
display(df_phase)

print("\n========== Endoscapes per-CVS AP ==========")
display(df_endo)

print("\n========== CholecT50 IVT per-triplet AP ==========")
display(df_t50_ivt)


# =========================================================
# 7. Sanity checks
# =========================================================

print("\n========== Sanity checks ==========")

cholec80_f1 = cholec80_overall_like_test(labels, preds, video_ids)
endo_map = endoscapes_overall_like_test(df_endo)
t50_ap_ivt = cholect50_ivt_overall_like_test(df_t50_ivt)

print("Cholec80 overall F1 like test:", cholec80_f1)
print("Endoscapes mAP from per-class AP:", endo_map)
print("CholecT50 AP-IVT from per-triplet AP:", t50_ap_ivt)


# =========================================================
# 8. Rare label views
# =========================================================

print("\n========== Rare Cholec80 phases ==========")
display(df_phase.sort_values("frequency"))

print("\n========== Rare Endoscapes CVS labels ==========")
display(df_endo.sort_values("frequency"))

print("\n========== Rare CholecT50 IVT triplets ==========")
display(df_t50_ivt.sort_values("frequency").head(20))


# =========================================================
# 9. Hardest label views
# =========================================================

print("\n========== Hardest Cholec80 phases ==========")
display(df_phase.sort_values("score").head(7))

print("\n========== Hardest Endoscapes CVS labels ==========")
display(df_endo.sort_values("score").head(3))

print("\n========== Hardest CholecT50 IVT triplets ==========")
display(df_t50_ivt.sort_values("score").head(20))


# =========================================================
# 10. Save CSV
# =========================================================

out_dir = "rebuttal_metric_analysis-(消融实验-LC-SNI)"
os.makedirs(out_dir, exist_ok=True)

df_phase.to_csv(
    os.path.join(out_dir, "cholec80_per_phase_f1.csv"),
    index=False
)

df_endo.to_csv(
    os.path.join(out_dir, "endoscapes_per_cvs_ap.csv"),
    index=False
)

df_t50_ivt.to_csv(
    os.path.join(out_dir, "cholect50_ivt_per_triplet_ap.csv"),
    index=False
)

print("\nSaved CSVs to:", out_dir)

# =========================================================
# 11. Print selected rare-label metrics for rebuttal
# =========================================================

print("\n\n==============================")
print("Selected rare-label metrics")
print("==============================")

# -----------------------------
# Cholec80: lowest-frequency 2 phases
# -----------------------------
phase_rare2 = (
    df_phase
    .sort_values("frequency", ascending=True)
    .head(2)
    .copy()
)

print("\n[Cholec80] Lowest-frequency 2 phases")
display(phase_rare2)

print(
    "[Cholec80] Rare-2 mean F1:",
    phase_rare2["score"].mean()
)


# -----------------------------
# Endoscapes: all 3 CVS labels sorted by frequency
# -----------------------------
endo_by_freq = (
    df_endo
    .sort_values("frequency", ascending=True)
    .copy()
)

print("\n[Endoscapes] CVS labels sorted by frequency")
display(endo_by_freq)

print(
    "[Endoscapes] Mean AP over 3 CVS labels:",
    endo_by_freq["score"].mean()
)


# -----------------------------
# CholecT50: lowest-frequency 10 IVT triplets
# -----------------------------
t50_rare10 = (
    df_t50_ivt
    .sort_values("frequency", ascending=True)
    .head(10)
    .copy()
)

print("\n[CholecT50] Lowest-frequency 10 IVT triplets")
display(t50_rare10)

print(
    "[CholecT50] Rare-10 mean AP:",
    t50_rare10["score"].mean()
)


# -----------------------------
# Also print compact summary table
# -----------------------------
summary_rows = [
    {
        "dataset": "Cholec80",
        "selected_labels": "lowest-frequency 2 phases",
        "metric": "F1",
        "num_labels": len(phase_rare2),
        "mean_score": phase_rare2["score"].mean(),
        "mean_frequency": phase_rare2["frequency"].mean(),
    },
    {
        "dataset": "Endoscapes",
        "selected_labels": "all 3 CVS labels",
        "metric": "AP",
        "num_labels": len(endo_by_freq),
        "mean_score": endo_by_freq["score"].mean(),
        "mean_frequency": endo_by_freq["frequency"].mean(),
    },
    {
        "dataset": "CholecT50",
        "selected_labels": "lowest-frequency 10 IVT triplets",
        "metric": "AP",
        "num_labels": len(t50_rare10),
        "mean_score": t50_rare10["score"].mean(),
        "mean_frequency": t50_rare10["frequency"].mean(),
    },
]

summary_rare = pd.DataFrame(summary_rows)

print("\n[Summary] Rare-label selected groups")
display(summary_rare)


# -----------------------------
# Save selected rare-label tables
# -----------------------------
phase_rare2.to_csv(
    os.path.join(out_dir, "selected_cholec80_lowest_freq_2_phases.csv"),
    index=False
)

endo_by_freq.to_csv(
    os.path.join(out_dir, "selected_endoscapes_all_3_by_frequency.csv"),
    index=False
)

t50_rare10.to_csv(
    os.path.join(out_dir, "selected_cholect50_lowest_freq_10_ivt.csv"),
    index=False
)

summary_rare.to_csv(
    os.path.join(out_dir, "selected_rare_label_summary.csv"),
    index=False
)

print("\nSaved selected rare-label CSVs to:", out_dir)
# =========================================================
# 12. Frequency-group summary: Low / Mid / High
# =========================================================

def make_frequency_group_summary(df, dataset_name, metric_name, group_col=None):
    """
    按 frequency 从低到高排序，然后分成 Low / Mid / High 三组。
    输出每组的平均 score、平均 frequency、frequency range。
    """

    df = df.copy()
    df = df.sort_values("frequency", ascending=True).reset_index(drop=True)

    groups = np.array_split(df, 3)
    group_names = ["Low", "Mid", "High"]

    rows = []

    for group_name, g in zip(group_names, groups):
        if len(g) == 0:
            continue

        row = {
            "dataset": dataset_name,
            "freq_group": group_name,
            "num_labels": len(g),
            "freq_min": int(g["frequency"].min()),
            "freq_max": int(g["frequency"].max()),
            "mean_frequency": float(g["frequency"].mean()),
            "metric": metric_name,
            "mean_score": float(g["score"].mean()),
        }

        if group_col is not None and group_col in g.columns:
            row["component"] = g[group_col].iloc[0]

        rows.append(row)

    return pd.DataFrame(rows)


# -----------------------------
# Cholec80: 7 phases -> Low/Mid/High
# -----------------------------
summary_phase_groups = make_frequency_group_summary(
    df_phase,
    dataset_name="Cholec80",
    metric_name="F1"
)

# -----------------------------
# Endoscapes: 3 CVS labels -> Low/Mid/High
# 每组 1 个 label
# -----------------------------
summary_endo_groups = make_frequency_group_summary(
    df_endo,
    dataset_name="Endoscapes",
    metric_name="AP"
)

# -----------------------------
# CholecT50: IVT triplets only
# 只看有效出现过的 triplet labels
# -----------------------------
summary_t50_groups = make_frequency_group_summary(
    df_t50_ivt,
    dataset_name="CholecT50-IVT",
    metric_name="AP",
    group_col="component"
)


# -----------------------------
# Combine all tasks
# -----------------------------
summary_freq_groups = pd.concat(
    [
        summary_phase_groups,
        summary_endo_groups,
        summary_t50_groups,
    ],
    ignore_index=True
)

print("\n========== Frequency-group summary: Low / Mid / High ==========")
display(summary_freq_groups)


# -----------------------------
# Save
# -----------------------------
summary_freq_groups.to_csv(
    os.path.join(out_dir, "frequency_group_summary_low_mid_high.csv"),
    index=False
)

print("\nSaved frequency-group summary to:", os.path.join(out_dir, "frequency_group_summary_low_mid_high.csv"))

In [ ]:
import os
import numpy as np
import pandas as pd
import warnings

from sklearn.metrics import f1_score, average_precision_score

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# =========================================================
# 0. 修改这里
# =========================================================

npz_path = "/data/MML1209/results/trial_scpnet_hill_Ignore_sp-comp-rebuttal/raw_predictions_110cls_epoch_3_mAP_6.11_loss_64.3053_ema.npz"

cholec80_cvs_gt_csv_path = "cholec80_cvs_gt_1fps_roi_endoscapes_order_or.csv"

out_dir = "cholec80_cvs_external_analysis-comp"
threshold = 0.5


# =========================================================
# 1. Load npz
# =========================================================

data = np.load(npz_path, allow_pickle=True)

preds = data["preds"]
labels = data["labels"]
video_ids = data["video_ids"]

labels = np.round(labels)

print("Loaded:", npz_path)
print("preds:", preds.shape)
print("labels:", labels.shape)
print("video_ids:", video_ids.shape)


# =========================================================
# 2. Helper functions
# =========================================================

def dataset_mask(video_ids, name):
    return np.array([name in str(v) for v in video_ids])


def safe_average_precision(y_true, y_score):
    if np.sum(y_true) == 0:
        return np.nan
    return average_precision_score(y_true, y_score) * 100


# =========================================================
# 3. Original Cholec80 phase metric
# =========================================================

def cholec80_original_phase_metric(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "cholec80")

    true = labels[mask][:, :7]
    pred = preds[mask][:, :7]
    vids = video_ids[mask]

    unique_vids = np.unique(vids)
    video_f1s = []

    for vid in unique_vids:
        idx = np.where(vids == vid)[0]

        y_true_video = np.argmax(true[idx], axis=1)
        y_pred_video = np.argmax(pred[idx], axis=1)

        f1 = f1_score(
            y_true_video,
            y_pred_video,
            average="macro",
            labels=np.unique(y_true_video),
        ) * 100

        video_f1s.append({
            "video_id": vid,
            "phase_f1": float(f1),
        })

    df_video = pd.DataFrame(video_f1s)

    summary = pd.DataFrame([{
        "metric": "Cholec80 original phase F1",
        "num_videos": int(len(df_video)),
        "score": float(df_video["phase_f1"].mean()),
    }])

    return summary, df_video


# =========================================================
# 4. Cholec80-CVS joint / conditional metrics
# =========================================================

def cholec80_cvs_joint_detail(labels, preds, video_ids, gt_csv_path, threshold=0.5):
    gt_df = pd.read_csv(gt_csv_path)

    gt_cols = [
        "C1_two_structures",
        "C2_hepatocystic_triangle",
        "C3_cystic_plate",
    ]

    mask = dataset_mask(video_ids, "cholec80")

    phase_true = np.argmax(labels[mask][:, :7], axis=1)
    phase_pred = np.argmax(preds[mask][:, :7], axis=1)

    cvs_score = preds[mask][:, 7:10]
    vids = np.array([str(v) for v in video_ids[mask]])

    pred_df = pd.DataFrame({
        "video_id": vids,
        "sec_1fps": pd.Series(vids).groupby(vids).cumcount().to_numpy(),
        "phase_true": phase_true,
        "phase_pred": phase_pred,
        "phase_correct": phase_true == phase_pred,
        "pred_C1_two_structures": cvs_score[:, 0],
        "pred_C2_hepatocystic_triangle": cvs_score[:, 1],
        "pred_C3_cystic_plate": cvs_score[:, 2],
    })

    merged = gt_df.merge(
        pred_df,
        on=["video_id", "sec_1fps"],
        how="inner",
    )

    print("Common samples:", merged.shape[0])

    true = merged[gt_cols].to_numpy(dtype=int)
    score = merged[
        [
            "pred_C1_two_structures",
            "pred_C2_hepatocystic_triangle",
            "pred_C3_cystic_plate",
        ]
    ].to_numpy()

    pred_bin = (score >= threshold).astype(int)
    phase_correct = merged["phase_correct"].to_numpy(dtype=bool)

    rows = []

    for subset_name, subset_mask in [
        ("all_common", np.ones(len(merged), dtype=bool)),
        ("phase_correct_only", phase_correct),
    ]:
        for cls, name in enumerate(gt_cols):
            ap = safe_average_precision(
                true[subset_mask, cls],
                score[subset_mask, cls],
            )

            binary_acc = (
                pred_bin[subset_mask, cls] == true[subset_mask, cls]
            ).mean() * 100

            rows.append({
                "subset": subset_name,
                "criterion": name,
                "frames": int(subset_mask.sum()),
                "positives": int(true[subset_mask, cls].sum()),
                "metric": "AP",
                "score": float(ap) if not np.isnan(ap) else np.nan,
                "binary_acc_at_threshold": float(binary_acc),
            })

    df_detail = pd.DataFrame(rows)

    map_by_subset = (
        df_detail
        .groupby("subset")["score"]
        .mean()
        .reset_index(name="CVS_mAP")
    )

    cvs_exact_correct = (pred_bin == true).all(axis=1)
    joint_exact_correct = phase_correct & cvs_exact_correct

    cvs_exact_given_phase_correct = (
        cvs_exact_correct[phase_correct].mean() * 100
        if np.sum(phase_correct) > 0
        else np.nan
    )

    summary = pd.DataFrame([{
        "common_frames": int(len(merged)),
        "phase_correct_frames": int(phase_correct.sum()),
        "phase_acc_on_common_frames": float(phase_correct.mean() * 100),
        "cvs_exact_acc_all_common": float(cvs_exact_correct.mean() * 100),
        "cvs_exact_acc_given_phase_correct": float(cvs_exact_given_phase_correct),
        "joint_phase_and_cvs_exact_acc": float(joint_exact_correct.mean() * 100),
        "threshold": threshold,
    }])

    return df_detail, map_by_subset, summary, merged


# =========================================================
# 5. Run
# =========================================================

df_phase_summary, df_phase_video = cholec80_original_phase_metric(
    labels,
    preds,
    video_ids,
)

df_c80_cvs, df_map_by_subset, df_joint_summary, df_common = cholec80_cvs_joint_detail(
    labels,
    preds,
    video_ids,
    cholec80_cvs_gt_csv_path,
    threshold=threshold,
)

print("\n========== Cholec80 original phase metric ==========")
print(df_phase_summary)

print("\n========== Cholec80 per-video phase F1 ==========")
print(df_phase_video)

print("\n========== Cholec80-CVS AP ==========")
print(df_c80_cvs)

print("\n========== CVS mAP by subset ==========")
print(df_map_by_subset)

print("\n========== Joint summary ==========")
print(df_joint_summary)


# =========================================================
# 6. Save
# =========================================================

os.makedirs(out_dir, exist_ok=True)

df_phase_summary.to_csv(
    os.path.join(out_dir, "cholec80_original_phase_summary.csv"),
    index=False,
)

df_phase_video.to_csv(
    os.path.join(out_dir, "cholec80_original_phase_per_video.csv"),
    index=False,
)

df_c80_cvs.to_csv(
    os.path.join(out_dir, "cholec80_cvs_external_joint_ap.csv"),
    index=False,
)

df_map_by_subset.to_csv(
    os.path.join(out_dir, "cholec80_cvs_external_map_by_subset.csv"),
    index=False,
)

df_joint_summary.to_csv(
    os.path.join(out_dir, "cholec80_cvs_external_joint_summary.csv"),
    index=False,
)

df_common.to_csv(
    os.path.join(out_dir, "cholec80_cvs_external_common_frames.csv"),
    index=False,
)

print("\nSaved CSVs to:", out_dir)


In [ ]:
import os
import numpy as np
import pandas as pd
import warnings

from sklearn.metrics import f1_score, average_precision_score

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# =========================================================
# 0. 修改这里
# =========================================================

npz_path = "/data/MML1209/results/trial_scpnet_hill_Ignore_sp_surgedino/raw_predictions_110cls_epoch_3_mAP_5.79_loss_64.8492_ema.npz"

cholec80_cvs_gt_csv_path = "cholec80_cvs_gt_1fps_roi_endoscapes_order_or.csv"

out_dir = "cholec80_cvs_external_analysis-surgenetdino"
threshold = 0.5


# =========================================================
# 1. Load npz
# =========================================================

data = np.load(npz_path, allow_pickle=True)

preds = data["preds"]
labels = data["labels"]
video_ids = data["video_ids"]

labels = np.round(labels)

print("Loaded:", npz_path)
print("preds:", preds.shape)
print("labels:", labels.shape)
print("video_ids:", video_ids.shape)


# =========================================================
# 2. Helper functions
# =========================================================

def dataset_mask(video_ids, name):
    return np.array([name in str(v) for v in video_ids])


def safe_average_precision(y_true, y_score):
    if np.sum(y_true) == 0:
        return np.nan
    return average_precision_score(y_true, y_score) * 100


# =========================================================
# 3. Original Cholec80 phase metric
# =========================================================

def cholec80_original_phase_metric(labels, preds, video_ids):
    mask = dataset_mask(video_ids, "cholec80")

    true = labels[mask][:, :7]
    pred = preds[mask][:, :7]
    vids = video_ids[mask]

    unique_vids = np.unique(vids)
    video_f1s = []

    for vid in unique_vids:
        idx = np.where(vids == vid)[0]

        y_true_video = np.argmax(true[idx], axis=1)
        y_pred_video = np.argmax(pred[idx], axis=1)

        f1 = f1_score(
            y_true_video,
            y_pred_video,
            average="macro",
            labels=np.unique(y_true_video),
        ) * 100

        video_f1s.append({
            "video_id": vid,
            "phase_f1": float(f1),
        })

    df_video = pd.DataFrame(video_f1s)

    summary = pd.DataFrame([{
        "metric": "Cholec80 original phase F1",
        "num_videos": int(len(df_video)),
        "score": float(df_video["phase_f1"].mean()),
    }])

    return summary, df_video


# =========================================================
# 4. Cholec80-CVS joint / conditional metrics
# =========================================================

def cholec80_cvs_joint_detail(labels, preds, video_ids, gt_csv_path, threshold=0.5):
    gt_df = pd.read_csv(gt_csv_path)

    gt_cols = [
        "C1_two_structures",
        "C2_hepatocystic_triangle",
        "C3_cystic_plate",
    ]

    mask = dataset_mask(video_ids, "cholec80")

    phase_true = np.argmax(labels[mask][:, :7], axis=1)
    phase_pred = np.argmax(preds[mask][:, :7], axis=1)

    cvs_score = preds[mask][:, 7:10]
    vids = np.array([str(v) for v in video_ids[mask]])

    pred_df = pd.DataFrame({
        "video_id": vids,
        "sec_1fps": pd.Series(vids).groupby(vids).cumcount().to_numpy(),
        "phase_true": phase_true,
        "phase_pred": phase_pred,
        "phase_correct": phase_true == phase_pred,
        "pred_C1_two_structures": cvs_score[:, 0],
        "pred_C2_hepatocystic_triangle": cvs_score[:, 1],
        "pred_C3_cystic_plate": cvs_score[:, 2],
    })

    merged = gt_df.merge(
        pred_df,
        on=["video_id", "sec_1fps"],
        how="inner",
    )

    print("Common samples:", merged.shape[0])

    true = merged[gt_cols].to_numpy(dtype=int)
    score = merged[
        [
            "pred_C1_two_structures",
            "pred_C2_hepatocystic_triangle",
            "pred_C3_cystic_plate",
        ]
    ].to_numpy()

    pred_bin = (score >= threshold).astype(int)
    phase_correct = merged["phase_correct"].to_numpy(dtype=bool)

    rows = []

    for subset_name, subset_mask in [
        ("all_common", np.ones(len(merged), dtype=bool)),
        ("phase_correct_only", phase_correct),
    ]:
        for cls, name in enumerate(gt_cols):
            ap = safe_average_precision(
                true[subset_mask, cls],
                score[subset_mask, cls],
            )

            binary_acc = (
                pred_bin[subset_mask, cls] == true[subset_mask, cls]
            ).mean() * 100

            rows.append({
                "subset": subset_name,
                "criterion": name,
                "frames": int(subset_mask.sum()),
                "positives": int(true[subset_mask, cls].sum()),
                "metric": "AP",
                "score": float(ap) if not np.isnan(ap) else np.nan,
                "binary_acc_at_threshold": float(binary_acc),
            })

    df_detail = pd.DataFrame(rows)

    map_by_subset = (
        df_detail
        .groupby("subset")["score"]
        .mean()
        .reset_index(name="CVS_mAP")
    )

    cvs_exact_correct = (pred_bin == true).all(axis=1)
    joint_exact_correct = phase_correct & cvs_exact_correct

    cvs_exact_given_phase_correct = (
        cvs_exact_correct[phase_correct].mean() * 100
        if np.sum(phase_correct) > 0
        else np.nan
    )

    summary = pd.DataFrame([{
        "common_frames": int(len(merged)),
        "phase_correct_frames": int(phase_correct.sum()),
        "phase_acc_on_common_frames": float(phase_correct.mean() * 100),
        "cvs_exact_acc_all_common": float(cvs_exact_correct.mean() * 100),
        "cvs_exact_acc_given_phase_correct": float(cvs_exact_given_phase_correct),
        "joint_phase_and_cvs_exact_acc": float(joint_exact_correct.mean() * 100),
        "threshold": threshold,
    }])

    return df_detail, map_by_subset, summary, merged


# =========================================================
# 5. Run
# =========================================================

df_phase_summary, df_phase_video = cholec80_original_phase_metric(
    labels,
    preds,
    video_ids,
)

df_c80_cvs, df_map_by_subset, df_joint_summary, df_common = cholec80_cvs_joint_detail(
    labels,
    preds,
    video_ids,
    cholec80_cvs_gt_csv_path,
    threshold=threshold,
)

print("\n========== Cholec80 original phase metric ==========")
print(df_phase_summary)

print("\n========== Cholec80 per-video phase F1 ==========")
print(df_phase_video)

print("\n========== Cholec80-CVS AP ==========")
print(df_c80_cvs)

print("\n========== CVS mAP by subset ==========")
print(df_map_by_subset)

print("\n========== Joint summary ==========")
print(df_joint_summary)


# =========================================================
# 6. Save
# =========================================================

os.makedirs(out_dir, exist_ok=True)

df_phase_summary.to_csv(
    os.path.join(out_dir, "cholec80_original_phase_summary.csv"),
    index=False,
)

df_phase_video.to_csv(
    os.path.join(out_dir, "cholec80_original_phase_per_video.csv"),
    index=False,
)

df_c80_cvs.to_csv(
    os.path.join(out_dir, "cholec80_cvs_external_joint_ap.csv"),
    index=False,
)

df_map_by_subset.to_csv(
    os.path.join(out_dir, "cholec80_cvs_external_map_by_subset.csv"),
    index=False,
)

df_joint_summary.to_csv(
    os.path.join(out_dir, "cholec80_cvs_external_joint_summary.csv"),
    index=False,
)

df_common.to_csv(
    os.path.join(out_dir, "cholec80_cvs_external_common_frames.csv"),
    index=False,
)

print("\nSaved CSVs to:", out_dir)
